In [0]:
from pyspark.sql.functions import lit

customer_trg = spark.read.table("main.ecommerce.customers_trg")
customer_SCD_df = (customer_trg
                   .withColumn("is_active", lit(True))
                   .withColumn("start_date", lit("2025-01-01"))
                   .withColumn("end_date",lit("9999-12-31"))
                   )

display(customer_SCD_df)


In [0]:
%sql
-- Insert into main.ecommerce.customers(customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state) values ('101','1993kkshbfk97',1098,'Pune','MH');

update main.ecommerce.customers set customer_state = 'MH', customer_city = 'Pune' where customer_id = 'dc5e67641a1399fb3d2fe80769d85520';

--select * from main.ecommerce.customers

In [0]:
customer_src = spark.read.table("main.ecommerce.customers")
#customer_src.printSchema()
#display(customer_src)

In [0]:
from pyspark.sql.functions import col

# Step 1: filter current records 
current_target_df = customer_SCD_df.filter(col("is_active") == True)
display(current_target_df)

In [0]:
# step 2 : Join SRC with current TRG
joined_df = customer_src.alias("src").join(current_target_df.alias("trg"), on = "customer_id", how = "left")
display(joined_df)

In [0]:
# Step:3 Detect changed records
changed_df = joined_df.filter(col("trg.customer_id").isNotNull() &
    (
        (col("src.customer_zip_code_prefix") != col("trg.customer_zip_code_prefix")) |
        (col("src.customer_city") != col("trg.customer_city")) |
        (col("src.customer_state") != col("trg.customer_state"))
    )
    )

display(changed_df)

In [0]:
# Step 4: expire old versions
from pyspark.sql.functions import *

expired_df = (changed_df
              .select("trg.*")
              .withColumn("is_active", lit(False))
              .withColumn("end_date", current_date())
) 

display(expired_df)

In [0]:
#Step 5: new versions of chnaged records

new_version_df = (changed_df
                  .select(("src.*"), lit(True).alias("is_active"), current_date().alias("start_date"), lit("9999-12-31").alias("end_date")
                  )
)

display(new_version_df)

In [0]:
#Step 6: Identify New records

new_records_df = joined_df.filter(col("trg.customer_id").isNull())\
                .select(("src.*"), lit(True).alias("is_active"), current_date().alias("start_date"), lit("9999-01-01").alias("end_date"))
            
display(new_records_df)


In [0]:
# step 7: keep unchnaged records AS-IS
unchanged_df = current_target_df.join(changed_df,on = 'customer_id', how= 'leftanti')

display(unchanged_df)

In [0]:
# Step 8: Final SCD dim

final_customer_dim_df = expired_df.union(new_version_df).union(new_records_df).union(unchanged_df)

display(final_customer_dim_df)


In [0]:
final_customer_dim_df.write.format("delta").mode("overwrite").saveAsTable("main.ecommerce.final_SCD_customer_dim")